## Sentence-BERT + Logistic Regression Accuracy

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import pandas as pd

# Load Banking77 dataset
dataset = load_dataset("banking77")
train_data = dataset["train"]
test_data = dataset["test"]

# No OOS filtering needed (Banking77 has only in-scope intents)
train_in_scope = train_data
test_in_scope = test_data

# Load Sentence-BERT
model = SentenceTransformer('intfloat/e5-base')

# Generate embeddings
train_texts = [ex["text"] for ex in train_in_scope]
test_texts = [ex["text"] for ex in test_in_scope]
train_embeddings = model.encode(train_texts, batch_size=64, show_progress_bar=True)
test_embeddings = model.encode(test_texts, batch_size=64, show_progress_bar=True)

# Get intent labels (Banking77 uses indices 0-76, convert to names for consistency)
intent_names = dataset["train"].features["label"].names
train_labels = [intent_names[ex["label"]] for ex in train_in_scope]
test_labels = [intent_names[ex["label"]] for ex in test_in_scope]

# Train logistic regression
clf = LogisticRegression(multi_class='multinomial', max_iter=1000)
clf.fit(train_embeddings, train_labels)



Batches:   0%|          | 0/157 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Predict and evaluate
model = SentenceTransformer('intfloat/e5-base')
test_in_scope = test_data
test_texts = [ex["text"] for ex in test_in_scope]
test_embeddings = model.encode(test_texts, batch_size=64, show_progress_bar=True)
# Get intent labels (Banking77 uses indices 0-76, convert to names for consistency)
intent_names = dataset["train"].features["label"].names
test_labels = [intent_names[ex["label"]] for ex in test_in_scope]
predictions = clf.predict(test_embeddings)
accuracy = accuracy_score(test_labels, predictions)
print(f"Sentence-BERT + Logistic Regression Accuracy on Banking77: {accuracy:.4f}")
# Save predictions to CSV
results_df = pd.DataFrame({
    "text": test_texts,
    "true_label": test_labels,
    "predicted_label": predictions
})

results_df.to_csv("predictions.csv", index=False)
print("Predictions saved to predictions.csv")


Batches:   0%|          | 0/49 [00:00<?, ?it/s]

Sentence-BERT + Logistic Regression Accuracy on Banking77: 0.8766
Predictions saved to predictions.csv


### regression_model_e5-base.pkl

In [ ]:
import pickle

# Save model
with open('banking77_regression_model_e5-base.pkl', 'wb') as f:
    pickle.dump(clf, f)


## Support Vector Classification.

## IntentEmbeddings

In [ ]:
import torch
torch.cuda.empty_cache()
from sentence_transformers import SentenceTransformer
from typing import Dict
import numpy as np
from functools import lru_cache

class SentenceEmbedder:
    def __init__(self):
        # Load a pre-trained Sentence-BERT model
        self.model = SentenceTransformer('intfloat/e5-base')  # Lightweight and effective

    def get_embedding(self, text):
       """
        Generate embedding for a given text, always prepending 'query: '.

        Args:
            text (str): The input string (treated as a query).

        Returns:
            np.ndarray: The embedding vector.
        """
        # Always prepend 'query: ' to the text
       formatted_text = "query: " + text.strip()
       return self.model.encode(formatted_text)


class IntentEmbeddings:
    def __init__(self, intents):
        self.intents = intents
        self.embedder = SentenceEmbedder()
        self.intent_embeddings = self.compute_intent_embeddings()

    def compute_intent_embeddings(self):
        """Generate embeddings for all intents."""
        embeddings = {}
        for intent, description in self.intents.items():
            combined_text = f"{intent}: {description}"  # Combine intent name and description
            embeddings[intent] = self.embedder.get_embedding(combined_text)
        return embeddings

## SequenceMatcher

## Customer agent

In [ ]:
import osfrom openai import OpenAIfrom dotenv import load_dotenvimport timeload_dotenv()import osfrom dotenv import load_dotenvload_dotenv()api_key = os.getenv("OPENAI_API_KEY")client = OpenAI(api_key=api_key)RATE_LIMIT_DELAY = 10MODEL = "gpt-4o-mini"def customer_agent(conversation_history, chatbot_question):    system_prompt = f"""You are playing the role of a real human user chatting with a support chatbot.This is your conversation so far:{conversation_history}Now the chatbot is trying to understand your intent. It may ask you to choose between two or three possible options, or clarify your issue.Respond like a real person would. Sometimes people are clear and direct and may choose option they find reasonable, sometimes they’re informal or unsure. Use your own words.It's okay to be casual, make typos,or ramble a bit sometimes. Vary your behavior the way real users do.Do not keep repreating same question again and again. Behave like user who wants to make chatbot understand their query.If  chatbot responds with something completely irrelevant to your question inform it so."""    messages = [        {"role": "system", "content": system_prompt},        {"role": "user", "content": chatbot_question}    ]    time.sleep(RATE_LIMIT_DELAY)    response = client.chat.completions.create(        model=MODEL,        messages=messages,        temperature=0.01    )    return response.choices[0].message.content.strip()

## DSMassFunction

In [ ]:
import torch
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from difflib import SequenceMatcher
from transformers import AutoModel, AutoTokenizer
import numpy as np
from datasets import load_dataset

# --- Setup ---
dataset = load_dataset("banking77")

# Intent mapping
intent_feature = dataset["train"].features["label"]
# index_to_name example: {0: 'restaurant_reviews', 1: 'nutrition_info', ..., 149: 'last_intent'}
index_to_name = {i: name for i, name in enumerate(intent_feature.names) if name != "oos"}

class DSMassFunction:
    def __init__(self, intent_embeddings, hierarchy, custom_thresholds=None):
        self.intent_embeddings = intent_embeddings
        self.embedder = SentenceEmbedder()
        self.hierarchy = hierarchy
        self.custom_thresholds = custom_thresholds or {}
        self.conversation_history = []
        self.user_response = None
        self.clf = clf
        self.index_to_name = index_to_name

    def is_leaf(self, intent):
        """Check if a node is a leaf in the hierarchy."""
        return len(self.hierarchy.get(intent, [])) == 0

    def get_threshold(self, intent):
        """Get threshold based on the level of the intent in the hierarchy."""
        if self.is_leaf(intent):
            return 0.1
        elif intent in self.hierarchy:
            return 0.2
        else:
            return 0.3

    def get_confidence_threshold(self, intent):
        """Get confidence threshold for an intent."""
        if intent in self.custom_thresholds:
            return self.custom_thresholds[intent]
        elif self.is_leaf(intent):
            return 0.3
        elif intent in self.hierarchy:
            return 0.5
        else:
            return 0.7

    def get_node_depth(self, node):
        """Compute the depth of a node in the hierarchy."""
        depth = 0
        current = node
        while current in self.hierarchy and self.hierarchy[current]:
            depth += 1
            current = self.hierarchy[current][0]
        return depth

    def find_lowest_common_ancestor(self, nodes):
        """Find the lowest common ancestor (LCA) of given nodes."""
        if not nodes:
            return None

        ancestors = []
        for node in nodes:
            node_ancestors = set()
            current = node
            while True:
                node_ancestors.add(current)
                parent = next(
                    (parent for parent, children in self.hierarchy.items() if current in children), None
                )
                if parent is None:
                    break
                current = parent
            ancestors.append(node_ancestors)

        common_ancestors = set.intersection(*ancestors)
        lca, min_depth = None, float("inf")
        for ancestor in common_ancestors:
            depth = self.get_node_depth(ancestor)
            if depth < min_depth:
                min_depth = depth
                lca = ancestor
        return lca

    def compute_mass_function(self, user_query):
        """Compute normalized Dempster-Shafer mass function using clf probabilities."""
        self.conversation_history.append(f"User: {user_query}")
        query_embedding = self.embedder.get_embedding(user_query)
        probs = self.clf.predict_proba(query_embedding.reshape(1, -1))[0]

        mass_function = {intent: 0.0 for intent in self.intent_embeddings.keys()}
        for intent_name, prob  in zip(self.clf.classes_, probs):
            if intent_name in mass_function:
                mass_function[intent_name] = prob

        total_mass = sum(mass_function.values())
        if total_mass > 0:
            mass_function = {intent: mass / total_mass for intent, mass in mass_function.items()}
        else:
            mass_function = {intent: 0.0 for intent in self.intent_embeddings.keys()}
            mass_function["Uncertainty"] = 1.0
        return mass_function

    def compute_belief(self, mass_function):
        """Compute belief values for all intents bottom-up."""
        belief = {}

        def compute_node_belief(intent):
            if intent in belief:
                return belief[intent]
            node_belief = mass_function.get(intent, 0)
            children = self.hierarchy.get(intent, [])
            for child in children:
                node_belief += compute_node_belief(child)
            belief[intent] = node_belief
            return node_belief

        for intent in self.hierarchy:
            compute_node_belief(intent)
        return belief

    def combine_mass_functions(self, mass1, mass2):
        """Combine two mass functions using Dempster's rule."""
        combined_mass = {}
        conflict = 0
        for a in mass1:
            for b in mass2:
                hcd = self.find_highest_common_descendant(a, b)
                intersection = hcd if hcd else "Uncertainty"
                contribution = mass1[a] * mass2[b]
                if intersection == "Uncertainty":
                    conflict += contribution
                else:
                    combined_mass[intersection] = combined_mass.get(intersection, 0) + contribution
        if conflict < 1:
            for key in combined_mass:
                combined_mass[key] /= (1 - conflict)
        return combined_mass

    def find_highest_common_descendant(self, node1, node2):
        """Find the highest common descendant (HCD) of two nodes."""
        descendants1 = self.get_all_descendants(node1)
        descendants2 = self.get_all_descendants(node2)
        common_descendants = descendants1.intersection(descendants2)
        if not common_descendants:
            return None
        hcd, max_depth = None, -1
        for descendant in common_descendants:
            depth = self.get_node_depth(descendant)
            if depth > max_depth:
                max_depth = depth
                hcd = descendant
        return hcd

    def get_all_descendants(self, node):
        """Get all descendants of a node in the hierarchy."""
        descendants = set()
        stack = [node]
        while stack:
            current = stack.pop()
            descendants.add(current)
            children = self.hierarchy.get(current, [])
            for child in children:
                stack.append(child)
        return descendants

    def evaluate_hierarchy(self, nodes, mass_function):
        """Return confident nodes and belief values."""
        belief = self.compute_belief(mass_function)
        confident_nodes = []
        for intent in nodes:
            intent_belief = belief.get(intent, 0)
            threshold = self.get_confidence_threshold(intent)
            if intent_belief >= threshold:
                confident_nodes.append((intent, intent_belief))
        return confident_nodes, belief

    def ask_clarification(self, parent_nodes, belief):
        """Generate clarification queries for ambiguous parent nodes."""
        clarification_queries = []
        for parent, _ in parent_nodes:
            children = self.hierarchy.get(parent, [])
            if len(children) < 4:
                clarification_queries.append((parent, children))
            else:
                children_with_belief = [(child, belief.get(child, 0)) for child in children]
                children_with_belief.sort(key=lambda x: x[1], reverse=True)
                top_children = [child for child, _ in children_with_belief[:3]]
                clarification_queries.append((parent, top_children))
        return clarification_queries

    def evaluate_with_clarifications(self, initial_mass, depth=0, maximum_depth=5):
        """Evaluate recursively with clarification queries until a confident leaf is found."""
        if depth >= maximum_depth:
            return None, None

        def evaluate_from_leaves(current_mass, depth=0, maximum_depth=5):
            if depth >= maximum_depth:
                return None, None

            # Find all leaf nodes
            leaf_nodes = [intent for intent in self.hierarchy if self.is_leaf(intent)]
            confident_nodes, belief = self.evaluate_hierarchy(leaf_nodes, current_mass)

            if confident_nodes:
                # --- CASE 1: Confident leaf nodes ---
                confident_leaf_nodes = [n for n in confident_nodes if self.is_leaf(n[0])]
                if len(confident_leaf_nodes) == 1:
                    print(f"Single confident leaf node found: {confident_leaf_nodes[0][0]}")
                    return confident_leaf_nodes[0]

                if len(confident_leaf_nodes) > 1:
                    lca = self.find_lowest_common_ancestor([i for i, _ in confident_leaf_nodes])
                    if lca:
                        print(f"Multiple confident leaf nodes found. LCA: {lca}")
                        clarification_queries = self.ask_clarification([(lca, belief.get(lca, 0))], belief)
                        for parent, children in clarification_queries:
                            chatbot_question = (
                                f"It seems like you're looking for something related to {parent}. "
                                f"Could you clarify which specific thing you're interested in? "
                                f"Here are a few suggestions: ({children})"
                            )
                            self.conversation_history.append(f"Chatbot: {chatbot_question}")
                            self.user_response = customer_agent("\n".join(self.conversation_history), chatbot_question)
                            #self.conversation_history.append(f"User: {self.user_response}")
                            user_mass = self.compute_mass_function(self.user_response)
                            current_mass = self.combine_mass_functions(current_mass, user_mass)
                        return evaluate_from_leaves(current_mass, depth + 1, maximum_depth)
                    else:
                        top_nodes = sorted(confident_leaf_nodes, key=lambda x: x[1], reverse=True)[:3]
                        options = [i for i, _ in top_nodes]
                        chatbot_question = (
                            f"There are a few things that might match: ({options}). "
                            f"Could you clarify a bit more?"
                        )
                        self.conversation_history.append(f"Chatbot: {chatbot_question}")
                        self.user_response = customer_agent("\n".join(self.conversation_history), chatbot_question)
                        #self.conversation_history.append(f"User: {self.user_response}")
                        user_mass = self.compute_mass_function(self.user_response)
                        current_mass = self.combine_mass_functions(current_mass, user_mass)

                # --- CASE 2: No confident leaf nodes, handle non-leaf nodes ---
                else:
                    confident_non_leaf_nodes = [(i, v) for i, v in confident_nodes if not self.is_leaf(i)]
                    if confident_non_leaf_nodes:
                        print("confident non-leaf")
                        def height_from_leaves(node):
                            if self.is_leaf(node) or node not in self.hierarchy:
                                return 0
                            return 1 + max(height_from_leaves(c) for c in self.hierarchy[node])

                        heights = {i: height_from_leaves(i) for i, _ in confident_non_leaf_nodes}
                        min_height = min(heights.values())
                        lowest_nodes = [(i, v) for i, v in confident_non_leaf_nodes if heights[i] == min_height]

                        lca_non_leaf = self.find_lowest_common_ancestor([i for i, _ in lowest_nodes])
                        if lca_non_leaf:
                            print(f"LCA among lowest non-leaf nodes: {lca_non_leaf}")
                            clarification_queries = self.ask_clarification([(lca_non_leaf, belief.get(lca_non_leaf, 0))], belief)
                            for parent, children in clarification_queries:
                                chatbot_question = (
                                    f"It seems like you're looking for something related to {parent}. "
                                    f"Could you clarify which specific thing you're interested in? "
                                    f"Here are a few suggestions: ({children})"
                                )
                                self.conversation_history.append(f"Chatbot: {chatbot_question}")
                                self.user_response = customer_agent("\n".join(self.conversation_history), chatbot_question)
                                #self.conversation_history.append(f"User: {self.user_response}")
                                user_mass = self.compute_mass_function(self.user_response)
                                current_mass = self.combine_mass_functions(current_mass, user_mass)
                            return evaluate_from_leaves(current_mass, depth + 1, maximum_depth)
                        else:
                            options = [i for i, _ in lowest_nodes]
                            chatbot_question = (
                                f"There are several possibilities at this level: ({options}). "
                                f"Could you clarify which one fits best?"
                            )
                            self.conversation_history.append(f"Chatbot: {chatbot_question}")
                            self.user_response = customer_agent("\n".join(self.conversation_history), chatbot_question)
                            #self.conversation_history.append(f"User: {self.user_response}")
                            user_mass = self.compute_mass_function(self.user_response)
                            current_mass = self.combine_mass_functions(current_mass, user_mass)

            else:
                # --- CASE 3: No confident nodes ---
                chatbot_question = (
                    "I'm not entirely sure what you're asking. "
                    "Could you rephrase your question a bit?"
                )
                self.conversation_history.append(f"Chatbot: {chatbot_question}")
                self.user_response = customer_agent("\n".join(self.conversation_history), chatbot_question)
                #self.conversation_history.append(f"User: {self.user_response}")
                user_mass = self.compute_mass_function(self.user_response)
                current_mass = self.combine_mass_functions(current_mass, user_mass)

            # Recurse
            return evaluate_from_leaves(current_mass, depth + 1, maximum_depth)

        # Start evaluation
        return evaluate_from_leaves(initial_mass, depth=depth)


## hierarchy

In [ ]:
hierarchy = {
    "Card Issues": [
        "Physical Cards",
        "Virtual Cards"
    ],
    "Physical Cards": [
        "activate_my_card",
        "card_about_to_expire",
        "card_arrival",
        "card_delivery_estimate",
        "card_not_working",
        "card_swallowed",
        "get_physical_card",
        "getting_spare_card",
        "lost_or_stolen_card",
        "order_physical_card"
    ],
    "activate_my_card": [],
    "card_about_to_expire": [],
    "card_arrival": [],
    "card_delivery_estimate": [],
    "card_not_working": [],
    "card_swallowed": [],
    "get_physical_card": [],
    "getting_spare_card": [],
    "lost_or_stolen_card": [],
    "order_physical_card": [],
    "Virtual Cards": [
        "get_disposable_virtual_card",
        "getting_virtual_card",
        "virtual_card_not_working"
    ],
    "get_disposable_virtual_card": [],
    "getting_virtual_card": [],
    "virtual_card_not_working": [],

    "Card Payments": [
        "Payment Problems"
    ],
    "Payment Problems": [
        "card_acceptance",
        "card_linking",
        "card_payment_fee_charged",
        "card_payment_not_recognised",
        "card_payment_wrong_exchange_rate",
        "contactless_not_working",
        "declined_card_payment",
        "disposable_card_limits",
        "pending_card_payment",
        "reverted_card_payment?",
        "direct_debit_payment_not_recognised",
        "transaction_charged_twice"
    ],
    "card_acceptance": [],
    "card_linking": [],
    "card_payment_fee_charged": [],
    "card_payment_not_recognised": [],
    "card_payment_wrong_exchange_rate": [],
    "contactless_not_working": [],
    "declined_card_payment": [],
    "disposable_card_limits": [],
    "pending_card_payment": [],
    "reverted_card_payment?": [],
    "direct_debit_payment_not_recognised": [],
    "transaction_charged_twice": [],

    "Payments and Transfers": [
        "Transfers",
        "Cash Withdrawals"
    ],
    "Transfers": [
        "beneficiary_not_allowed",
        "cancel_transfer",
        "declined_transfer",
        "failed_transfer",
        "pending_transfer",
        "transfer_fee_charged",
        "transfer_into_account",
        "transfer_not_received_by_recipient",
        "transfer_timing"
    ],
    "beneficiary_not_allowed": [],
    "cancel_transfer": [],
    "declined_transfer": [],
    "failed_transfer": [],
    "pending_transfer": [],
    "transfer_fee_charged": [],
    "transfer_into_account": [],
    "transfer_not_received_by_recipient": [],
    "transfer_timing": [],
    "Cash Withdrawals": [
        "atm_support",
        "cash_withdrawal_charge",
        "cash_withdrawal_not_recognised",
        "declined_cash_withdrawal",
        "pending_cash_withdrawal",
        "wrong_amount_of_cash_received",
        "wrong_exchange_rate_for_cash_withdrawal"
    ],
    "atm_support": [],
    "cash_withdrawal_charge": [],
    "cash_withdrawal_not_recognised": [],
    "declined_cash_withdrawal": [],
    "pending_cash_withdrawal": [],
    "wrong_amount_of_cash_received": [],
    "wrong_exchange_rate_for_cash_withdrawal": [],

    "Account Services": [
        "Account Management",
        "Top-Ups"
    ],
    "Account Management": [
        "balance_not_updated_after_bank_transfer",
        "balance_not_updated_after_cheque_or_cash_deposit",
        "edit_personal_details",
        "passcode_forgotten",
        "pin_blocked",
        "terminate_account"
    ],
    "balance_not_updated_after_bank_transfer": [],
    "balance_not_updated_after_cheque_or_cash_deposit": [],
    "edit_personal_details": [],
    "passcode_forgotten": [],
    "pin_blocked": [],
    "terminate_account": [],
    "Top-Ups": [
        "automatic_top_up",
        "pending_top_up",
        "top_up_by_bank_transfer_charge",
        "top_up_by_card_charge",
        "top_up_by_cash_or_cheque",
        "top_up_failed",
        "top_up_limits",
        "top_up_reverted",
        "topping_up_by_card",
        "verify_top_up"
    ],
    "automatic_top_up": [],
    "pending_top_up": [],
    "top_up_by_bank_transfer_charge": [],
    "top_up_by_card_charge": [],
    "top_up_by_cash_or_cheque": [],
    "top_up_failed": [],
    "top_up_limits": [],
    "top_up_reverted": [],
    "topping_up_by_card": [],
    "verify_top_up": [],

    "Security and Verification": [
        "Verification",
        "Security Issues"
    ],
    "Verification": [
        "unable_to_verify_identity",
        "verify_my_identity",
        "verify_source_of_funds",
        "why_verify_identity"
    ],
    "unable_to_verify_identity": [],
    "verify_my_identity": [],
    "verify_source_of_funds": [],
    "why_verify_identity": [],
    "Security Issues": [
        "compromised_card",
        "lost_or_stolen_phone"
    ],
    "compromised_card": [],
    "lost_or_stolen_phone": [],

    "Fees and Refunds": [
        "Charges and Fees",
        "Refunds"
    ],
    "Charges and Fees": [
        "exchange_charge",
        "extra_charge_on_statement"
    ],
    "exchange_charge": [],
    "extra_charge_on_statement": [],
    "Refunds": [
        "Refund_not_showing_up",
        "request_refund"
    ],
    "Refund_not_showing_up": [],
    "request_refund": [],

    "General": [
        "General Inquiries"
    ],
    "General Inquiries": [
        "age_limit",
        "apple_pay_or_google_pay",
        "change_pin",
        "country_support",
        "exchange_rate",
        "exchange_via_app",
        "fiat_currency_support",
        "receiving_money",
        "supported_cards_and_currencies",
        "visa_or_mastercard"
    ],
    "age_limit": [],
    "apple_pay_or_google_pay": [],
    "change_pin": [],
    "country_support": [],
    "exchange_rate": [],
    "exchange_via_app": [],
    "fiat_currency_support": [],
    "receiving_money": [],
    "supported_cards_and_currencies": [],
    "visa_or_mastercard": []
}

## hierarchical_intents

In [ ]:
hierarchical_intents = {
    "Card Issues": "Issues related to physical or virtual cards and their functionality.",
    "Physical Cards": "Concerns involving physical debit or credit cards.",
    "activate_my_card": "Help with activating a new physical card.",
    "card_about_to_expire": "What to do when your card is about to expire.",
    "card_arrival": "Information about expected delivery of your card.",
    "card_delivery_estimate": "Estimate of how long it takes for your card to arrive.",
    "card_not_working": "Troubleshooting issues with card functionality.",
    "card_swallowed": "What to do if an ATM swallows your card.",
    "get_physical_card": "Requesting a physical card for your account.",
    "getting_spare_card": "Getting a replacement or additional physical card.",
    "lost_or_stolen_card": "Report and handle lost or stolen physical cards.",
    "order_physical_card": "Ordering a new or replacement physical card.",
    "Virtual Cards": "Support for virtual or disposable cards.",
    "get_disposable_virtual_card": "Getting a disposable virtual card.",
    "getting_virtual_card": "How to request or access a virtual card.",
    "virtual_card_not_working": "Problems with using virtual cards.",

    "Card Payments": "Help with payments made using a card.",
    "Payment Problems": "Issues like declined transactions or incorrect fees.",
    "card_acceptance": "Merchant declined card or doesn't accept it.",
    "card_linking": "Linking your card to apps or accounts.",
    "card_payment_fee_charged": "Unexpected fees on card payments.",
    "card_payment_not_recognised": "Disputing an unknown card payment.",
    "card_payment_wrong_exchange_rate": "Incorrect currency exchange rate applied.",
    "contactless_not_working": "Problems with using contactless feature.",
    "declined_card_payment": "Card payment was declined unexpectedly.",
    "disposable_card_limits": "Limits on how disposable cards can be used.",
    "pending_card_payment": "Card transaction pending or delayed.",
    "reverted_card_payment?": "A payment was reverted or refunded.",
    "direct_debit_payment_not_recognised": "Unknown or suspicious direct debit.",
    "transaction_charged_twice": "Same transaction charged multiple times.",

    "Payments and Transfers": "Issues with money transfers and withdrawals.",
    "Transfers": "Managing bank transfers and their statuses.",
    "beneficiary_not_allowed": "Cannot send money to selected beneficiary.",
    "cancel_transfer": "How to cancel a bank transfer.",
    "declined_transfer": "Transfer was declined or failed.",
    "failed_transfer": "Transfer did not complete successfully.",
    "pending_transfer": "Transfer is pending or in process.",
    "transfer_fee_charged": "Unexpected transfer fees.",
    "transfer_into_account": "Receiving transfers into your account.",
    "transfer_not_received_by_recipient": "Recipient hasn't received the money.",
    "transfer_timing": "How long transfers usually take.",
    "Cash Withdrawals": "Withdrawing cash and related issues.",
    "atm_support": "Using ATMs and their availability.",
    "cash_withdrawal_charge": "Fees when withdrawing money.",
    "cash_withdrawal_not_recognised": "Unknown or suspicious cash withdrawal.",
    "declined_cash_withdrawal": "ATM withdrawal was declined.",
    "pending_cash_withdrawal": "Withdrawal is still processing.",
    "wrong_amount_of_cash_received": "Incorrect cash dispensed by ATM.",
    "wrong_exchange_rate_for_cash_withdrawal": "Poor currency conversion on withdrawal.",

    "Account Services": "Managing your account, top-ups, and profile.",
    "Account Management": "Profile updates and account access issues.",
    "balance_not_updated_after_bank_transfer": "Transfer completed but balance not updated.",
    "balance_not_updated_after_cheque_or_cash_deposit": "Cash/cheque deposit not reflected.",
    "edit_personal_details": "Update your name, address, etc.",
    "passcode_forgotten": "What to do if you forget your passcode.",
    "pin_blocked": "Your PIN is blocked — how to unblock or reset.",
    "terminate_account": "Close your account permanently.",
    "Top-Ups": "Adding money to your account.",
    "automatic_top_up": "Enable or troubleshoot automatic top-ups.",
    "pending_top_up": "Top-up is delayed or pending.",
    "top_up_by_bank_transfer_charge": "Charges for top-up by bank.",
    "top_up_by_card_charge": "Fees for top-up by debit/credit card.",
    "top_up_by_cash_or_cheque": "Add funds via cash or cheque.",
    "top_up_failed": "Top-up attempt failed.",
    "top_up_limits": "Limits on how much you can top up.",
    "top_up_reverted": "Top-up got reverted or cancelled.",
    "topping_up_by_card": "How to top up using a card.",
    "verify_top_up": "Verification required for a top-up.",

    "Security and Verification": "ID verification and security concerns.",
    "Verification": "Identity and source of funds checks.",
    "unable_to_verify_identity": "Identity could not be verified.",
    "verify_my_identity": "Steps to verify your identity.",
    "verify_source_of_funds": "Proof for where your money comes from.",
    "why_verify_identity": "Why verification is necessary.",
    "Security Issues": "Card or phone compromised.",
    "compromised_card": "Your card might have been compromised.",
    "lost_or_stolen_phone": "What to do if your phone is lost or stolen.",

    "Fees and Refunds": "Fee details and refund requests.",
    "Charges and Fees": "Unexpected charges and fees.",
    "exchange_charge": "Fee applied during currency exchange.",
    "extra_charge_on_statement": "Extra or unclear charges in statement.",
    "Refunds": "Issues with getting your money back.",
    "Refund_not_showing_up": "A refund hasn’t appeared in your account.",
    "request_refund": "How to request a refund.",

    "General": "General questions not tied to a transaction.",
    "General Inquiries": "Common questions about the service.",
    "age_limit": "Minimum age required to open an account.",
    "apple_pay_or_google_pay": "Using Apple Pay or Google Pay.",
    "change_pin": "How to change your card's PIN.",
    "country_support": "Supported countries for services.",
    "exchange_rate": "Current rates and how they apply.",
    "exchange_via_app": "How to convert currency within the app.",
    "fiat_currency_support": "Which fiat currencies are supported.",
    "receiving_money": "Can you receive money from others?",
    "supported_cards_and_currencies": "List of cards/currencies supported.",
    "visa_or_mastercard": "Card type — Visa or Mastercard."
}


## Optimal Thresholds

---



## load dataset

In [ ]:
custom_thresholds = {}

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np
from tqdm import tqdm
import json
import os



# --- Load Banking77 dataset ---
ds = load_dataset("banking77")
dataset = ds["test"]  # Use train split (10,003 examples)

# Override intent_names with correct intents
intent_names  = ds["train"].features["label"].names

print("Corrected Banking77 intent_names:", sorted(intent_names))

# Create intent map with corrected intents
intent_map = {i: name for i, name in enumerate(intent_names)}

# Format data, mapping labels to corrected intents
formatted_data = []
for ex in dataset:
    label = ex["label"]
    # Ensure label is within valid range (0–76)
    if 0 <= label < len(intent_names):
        formatted_data.append({"Example": ex["text"], "Label": intent_map[label]})
    else:
        print(f"Skipping invalid label {label} for query: {ex['text']}")

print(f"Generated formatted_data with {len(formatted_data)} examples")

# Verify formatted_data intents
formatted_intents = set(ex["Label"] for ex in formatted_data)
invalid_intents = formatted_intents - set(intent_names)
if invalid_intents:
    print(f"Error: Invalid intents in formatted_data: {invalid_intents}")
    formatted_data = [ex for ex in formatted_data if ex["Label"] in intent_names]
    print(f"Filtered formatted_data to {len(formatted_data)} examples with valid intents")


# --- Load hierarchy ---


# Validate hierarchy intents against corrected intent_names
hierarchy_intents = []
for category, intents in hierarchy.items():
    if isinstance(intents, list):
        hierarchy_intents.extend(intents)
hierarchy_intents_set = set(hierarchy_intents)
formatted_intents_set = set(ex["Label"] for ex in formatted_data)

# Check for mismatches
if hierarchy_intents_set != formatted_intents_set:
    print("Warning: Intent mismatch between hierarchy and formatted_data")
    print("Intents in hierarchy but not in data:", hierarchy_intents_set - formatted_intents_set)
    print("Intents in data but not in hierarchy:", formatted_intents_set - hierarchy_intents_set)
    formatted_data = [ex for ex in formatted_data if ex["Label"] in hierarchy_intents_set]
    print(f"Filtered formatted_data to {len(formatted_data)} examples with valid intents")
else:
    print("Hierarchy and formatted_data intents match perfectly!")

# --- Function to split a list into n equal parts ---
def chunk_data(data, n_chunks):
    avg_chunk_size = len(data) // n_chunks
    chunks = []
    for i in range(n_chunks):
        start_idx = i * avg_chunk_size
        end_idx = (i + 1) * avg_chunk_size if i != n_chunks - 1 else len(data)
        chunks.append(data[start_idx:end_idx])
    return chunks

# --- Initialize embedder and calculator ---
try:
    intent_embedder = IntentEmbeddings(hierarchy)
    print("IntentEmbedder intents:", sorted(intent_embedder.intent_embeddings.keys()))
except AttributeError:
    print("Warning: Could not access intent_embedder.intent_embeddings. Ensure it's initialized correctly.")
    intent_embedder = IntentEmbeddings(hierarchy)

# Initialize DSMassFunction with hierarchy intents to avoid corrupted intent_names
ds_calculator = DSMassFunction(intent_embedder.intent_embeddings, hierarchy)

# --- Create a dictionary to store belief values for all examples ---
belief_results = []

# --- Split the data into 4 parts ---
n_chunks = 4
chunks = chunk_data(formatted_data, n_chunks)

# --- Process each chunk of data ---
for i, chunk in enumerate(chunks):
    print(f"Processing chunk {i + 1}...")

    chunk_belief_results = []

    for row in tqdm(chunk, desc=f"Chunk {i + 1}", unit="row"):
        user_query = row["Example"]
        correct_intent = str(row["Label"])  # Ensure Python string
        # Safety check for invalid intents
        if correct_intent not in intent_names:
            print(f"Skipping query '{user_query}' with invalid intent '{correct_intent}'")
            continue

        try:
            mass_function = ds_calculator.compute_mass_function(user_query)
            if mass_function is None:
                print(f"Warning: No mass function for query: {user_query}")
                continue

            belief = ds_calculator.compute_belief(mass_function)

            if belief:
                chunk_belief_results.append({
                    "user_question": user_query,
                    "correct_intent": correct_intent,
                    "belief_values": belief
                })
            else:
                print(f"Warning: No belief values for query: {user_query}")

        except Exception as e:
            print(f"Error processing query '{user_query}': {e}")
            print(f"Correct intent: {correct_intent}")
            print("Skipping query to continue processing...")

    # --- Convert the chunk results to a DataFrame ---
    belief_df_chunk = pd.DataFrame(chunk_belief_results)

    # --- Save the results to a CSV file ---
    output_filename = f"logistic_B77_chunk_{i + 1}_test.csv"
    belief_df_chunk.to_csv(output_filename, index=False)

    print(f"Chunk {i + 1} saved to {output_filename}")

print("All chunks processed and saved successfully!")


Corrected Banking77 intent_names: ['Refund_not_showing_up', 'activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire', 'card_acceptance', 'card_arrival', 'card_delivery_estimate', 'card_linking', 'card_not_working', 'card_payment_fee_charged', 'card_payment_not_recognised', 'card_payment_wrong_exchange_rate', 'card_swallowed', 'cash_withdrawal_charge', 'cash_withdrawal_not_recognised', 'change_pin', 'compromised_card', 'contactless_not_working', 'country_support', 'declined_card_payment', 'declined_cash_withdrawal', 'declined_transfer', 'direct_debit_payment_not_recognised', 'disposable_card_limits', 'edit_personal_details', 'exchange_charge', 'exchange_rate', 'exchange_via_app', 'extra_charge_on_statement', 'failed_transfer', 'fiat_currency_support', 'get_disposable_virtual_card', 'get_p

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

Chunk 2 saved to logistic_B77_chunk_2_test.csv
Processing chunk 3...


Chunk 3: 100%|██████████| 770/770 [01:46<00:00,  7.21row/s]


Chunk 3 saved to logistic_B77_chunk_3_test.csv
Processing chunk 4...


Chunk 4: 100%|██████████| 770/770 [01:54<00:00,  6.71row/s]


Chunk 4 saved to logistic_B77_chunk_4_test.csv
All chunks processed and saved successfully!


## Optimal Thresholds

In [ ]:
custom_thresholds={}

### load regression_model_e5-base_0.88.pkl as clf

In [ ]:
import pickle
with open('banking77_regression_model_e5-base.pkl', 'rb') as f:
    clf = pickle.load(f)

### Merged all chunks into train_clinc_trial

In [ ]:
import pandas as pd

# File names for the chunks
chunk_files = [
    "logistic_B77_chunk_1.csv",
    "logistic_B77_chunk_2.csv",
    "logistic_B77_chunk_3.csv",
    "logistic_B77_chunk_4.csv"
]

# Load and concatenate all chunks
dfs = [pd.read_csv(file) for file in chunk_files]
merged_df = pd.concat(dfs, ignore_index=True)

# Save merged file (optional but useful)
merged_df.to_csv("logistic_B77_merged.csv", index=False)

print("Merged all chunks into logistic_B77_merged.csv")


Merged all chunks into logistic_B77_merged.csv


###  Parse the belief values

In [ ]:
import pandas as pd
import re

# Load merged CSV
df = pd.read_csv("logistic_B77_merged.csv")

def parse_belief_values(raw_str):
    pattern = r"'(.*?)': (?:np\.float64\()?([0-9.eE+-]+)\)?"
    matches = re.findall(pattern, raw_str)
    return {intent: float(val) for intent, val in matches}

# Parse the belief values safely
df["belief_values"] = df["belief_values"].apply(parse_belief_values)

# Convert belief_values dicts to a new DataFrame all at once
belief_df = df["belief_values"].apply(pd.Series).fillna(0.0)

# Merge into original and drop the raw column
df = pd.concat([df.drop(columns=["belief_values"]), belief_df], axis=1)

# Save the transformed DataFrame
df.to_csv("transformed_logistic_B77_merged.csv", index=False)

# Optional preview
print(df.head())

                                       user_question correct_intent  \
0                     I am still waiting on my card?   card_arrival   
1  What can I do if my card still hasn't arrived ...   card_arrival   
2  I have been waiting over a week. Is the card s...   card_arrival   
3  Can I track my card while it is in the process...   card_arrival   
4  How do I know if I will get my card, or if it ...   card_arrival   

   activate_my_card  card_about_to_expire  card_arrival  \
0          0.008382              0.017217      0.278698   
1          0.014414              0.045606      0.243564   
2          0.006643              0.018961      0.375849   
3          0.007910              0.019824      0.454469   
4          0.017731              0.055694      0.098505   

   card_delivery_estimate  card_not_working  card_swallowed  \
0                0.034000          0.021515        0.011764   
1                0.073635          0.031902        0.010254   
2                0.080548    

In [ ]:
import pandas as pd

# Load the original CSV
df = pd.read_csv("transformed_logistic_B77_merged.csv")

# Define or load hierarchy
# Example:
# with open("hierarchy.json") as f:
#     hierarchy = json.load(f)

# Get all unique intents
intents = set(hierarchy.keys())
for children in hierarchy.values():
    intents.update(children)
intents = sorted(intents)

# Function to get ancestors
def get_ancestors(intent, hierarchy):
    ancestors = set()
    stack = [intent]
    while stack:
        current = stack.pop()
        ancestors.add(current)
        for parent, children in hierarchy.items():
            if current in children:
                stack.append(parent)
    return ancestors

# Create a list of dictionaries for new columns
new_columns = []
for index, row in df.iterrows():
    correct_intent = row["correct_intent"]
    ancestors = get_ancestors(correct_intent, hierarchy)
    new_row = {f"is_correct_{intent}": int(intent in ancestors) for intent in intents}
    new_columns.append(new_row)

from collections import defaultdict

def get_descendants(intent, hierarchy):
    children = hierarchy.get(intent, [])
    all_descendants = set(children)
    for child in children:
        all_descendants.update(get_descendants(child, hierarchy))
    return all_descendants

# Step 2: For each ancestor, sum the values of its descendant columns
for ancestor in hierarchy.keys():
    descendants = get_descendants(ancestor, hierarchy)
    # Only include descendants that actually exist as columns in the DataFrame
    valid_descendants = [desc for desc in descendants if desc in df.columns]
    if valid_descendants:
        df[f"sum_descendants_{ancestor}"] = df[valid_descendants].sum(axis=1)
# Convert new columns to DataFrame and concatenate
new_df = pd.DataFrame(new_columns)
df = pd.concat([df.drop(columns=["correct_intent"]), new_df], axis=1)

# Save to new CSV
df.to_csv("updated_logistic_B77_merged.csv", index=False)
print(df.head())

                                       user_question  activate_my_card  \
0                     I am still waiting on my card?          0.008382   
1  What can I do if my card still hasn't arrived ...          0.014414   
2  I have been waiting over a week. Is the card s...          0.006643   
3  Can I track my card while it is in the process...          0.007910   
4  How do I know if I will get my card, or if it ...          0.017731   

   card_about_to_expire  card_arrival  card_delivery_estimate  \
0              0.017217      0.278698                0.034000   
1              0.045606      0.243564                0.073635   
2              0.018961      0.375849                0.080548   
3              0.019824      0.454469                0.094484   
4              0.055694      0.098505                0.047251   

   card_not_working  card_swallowed  get_physical_card  getting_spare_card  \
0          0.021515        0.011764           0.026296            0.026311   
1       

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score

# Load the dataset
df = pd.read_csv("updated_logistic_B77_merged.csv")

# Generate list of all intents from the columns
intents = sorted([col.replace("is_correct_", "") for col in df.columns if col.startswith("is_correct_")])

# Dictionary to store optimal thresholds and scores
optimal_results = {}

# Precompute thresholds from 0.00 to 1.00
thresholds = np.linspace(0, 1, 101)

# Iterate over each intent
for intent in intents:
    belief_values = df[intent].values
    ground_truth = df[f"is_correct_{intent}"].values

    best_threshold = 0.0
    best_weighted_f1 = 0.0

    for threshold in thresholds:
        predictions = (belief_values >= threshold).astype(int)
        weighted_f1 = f1_score(ground_truth, predictions, average="weighted")

        if weighted_f1 > best_weighted_f1:
            best_weighted_f1 = weighted_f1
            best_threshold = threshold

    optimal_results[intent] = {
        "threshold": best_threshold,
        "f1_score": best_weighted_f1
    }

# Print the optimal thresholds and scores
print("Optimal Thresholds and Weighted F1 Scores:")
for intent, result in optimal_results.items():
    print(f"{intent}: Threshold = {result['threshold']:.2f}, F1 Score = {result['f1_score']:.4f}")


Optimal Thresholds and Weighted F1 Scores:
Account Management: Threshold = 0.20, F1 Score = 0.9874
Account Services: Threshold = 0.30, F1 Score = 0.9703
Card Issues: Threshold = 0.34, F1 Score = 0.9793
Card Payments: Threshold = 0.30, F1 Score = 0.9673
Cash Withdrawals: Threshold = 0.22, F1 Score = 0.9900
Charges and Fees: Threshold = 0.23, F1 Score = 0.9960
Fees and Refunds: Threshold = 0.24, F1 Score = 0.9943
General: Threshold = 0.27, F1 Score = 0.9892
General Inquiries: Threshold = 0.27, F1 Score = 0.9892
Payment Problems: Threshold = 0.30, F1 Score = 0.9673
Payments and Transfers: Threshold = 0.34, F1 Score = 0.9644
Physical Cards: Threshold = 0.30, F1 Score = 0.9832
Refund_not_showing_up: Threshold = 0.24, F1 Score = 0.9977
Refunds: Threshold = 0.24, F1 Score = 0.9983
Security Issues: Threshold = 0.12, F1 Score = 0.9951
Security and Verification: Threshold = 0.18, F1 Score = 0.9936
Top-Ups: Threshold = 0.29, F1 Score = 0.9802
Transfers: Threshold = 0.29, F1 Score = 0.9708
Verific

In [ ]:
# Save the optimal thresholds and scores to a JSON file
with open("optimal_thresholds_logistic_banking", "w") as json_file:
    json.dump(optimal_results, json_file, indent=4)

# Print the saved thresholds and scores
print("optimal_thresholds_logistic_banking")
for intent, result in optimal_results.items():
    print(f"{intent}: Threshold = {result['threshold']:.2f}")

optimal_thresholds_logistic_banking
Account Management: Threshold = 0.20
Account Services: Threshold = 0.30
Card Issues: Threshold = 0.34
Card Payments: Threshold = 0.30
Cash Withdrawals: Threshold = 0.22
Charges and Fees: Threshold = 0.23
Fees and Refunds: Threshold = 0.24
General: Threshold = 0.27
General Inquiries: Threshold = 0.27
Payment Problems: Threshold = 0.30
Payments and Transfers: Threshold = 0.34
Physical Cards: Threshold = 0.30
Refund_not_showing_up: Threshold = 0.24
Refunds: Threshold = 0.24
Security Issues: Threshold = 0.12
Security and Verification: Threshold = 0.18
Top-Ups: Threshold = 0.29
Transfers: Threshold = 0.29
Verification: Threshold = 0.14
Virtual Cards: Threshold = 0.25
activate_my_card: Threshold = 0.13
age_limit: Threshold = 0.15
apple_pay_or_google_pay: Threshold = 0.13
atm_support: Threshold = 0.12
automatic_top_up: Threshold = 0.12
balance_not_updated_after_bank_transfer: Threshold = 0.14
balance_not_updated_after_cheque_or_cash_deposit: Threshold = 0.1

In [ ]:
import json

# Load the original JSON structure
with open('optimal_thresholds_logistic_banking', 'r') as f:
    full_data = json.load(f)

# Extract the simplified format
custom_thresholds = {intent: data["threshold"] for intent, data in full_data.items()}


# Print a few to preview
print(dict(list(custom_thresholds .items())))

{'Account Management': 0.2, 'Account Services': 0.3, 'Card Issues': 0.34, 'Card Payments': 0.3, 'Cash Withdrawals': 0.22, 'Charges and Fees': 0.23, 'Fees and Refunds': 0.24, 'General': 0.27, 'General Inquiries': 0.27, 'Payment Problems': 0.3, 'Payments and Transfers': 0.34, 'Physical Cards': 0.3, 'Refund_not_showing_up': 0.24, 'Refunds': 0.24, 'Security Issues': 0.12, 'Security and Verification': 0.18, 'Top-Ups': 0.29, 'Transfers': 0.29, 'Verification': 0.14, 'Virtual Cards': 0.25, 'activate_my_card': 0.13, 'age_limit': 0.15, 'apple_pay_or_google_pay': 0.13, 'atm_support': 0.12, 'automatic_top_up': 0.12, 'balance_not_updated_after_bank_transfer': 0.14, 'balance_not_updated_after_cheque_or_cash_deposit': 0.13, 'beneficiary_not_allowed': 0.19, 'cancel_transfer': 0.2, 'card_about_to_expire': 0.14, 'card_acceptance': 0.07, 'card_arrival': 0.18, 'card_delivery_estimate': 0.18, 'card_linking': 0.11, 'card_not_working': 0.15, 'card_payment_fee_charged': 0.19, 'card_payment_not_recognised': 0.

## Main

In [ ]:
def main():
    from datasets import load_dataset
    import pandas as pd

    # Initialize intent embeddings
    intent_embedder = IntentEmbeddings(hierarchical_intents)

    # Load dataset and define index_to_name
    dataset = load_dataset("banking77")
    intent_feature = dataset["train"].features["label"]
    index_to_name = {i: name for i, name in enumerate(intent_feature.names)}

    # Load and filter test data
    test_data = dataset["test"]
    test_data = [{"text": ex["text"], "label": ex["label"]} for ex in test_data]

    # Split into 10 chunks
    total = len(test_data)
    chunk_size = total // 10
    chunks = [test_data[i*chunk_size : (i+1)*chunk_size] for i in range(9)]
    chunks.append(test_data[9*chunk_size:])  # final chunk includes remainder
    print(chunks[0])
    ds_calculator = DSMassFunction(intent_embedder.intent_embeddings, hierarchy, custom_thresholds)
    # Process each chunk
    for idx, val_chunk in enumerate(chunks):
        print(f"\n--- Processing chunk {idx + 1}/10 ({len(val_chunk)} samples) ---")
        results = []
        for ex in val_chunk:
            user_query = ex["text"]
            true_intent_idx = ex["label"]
            true_intent = index_to_name.get(true_intent_idx, "unknown")
            ds_calculator.conversation_history = []
            initial_mass = ds_calculator.compute_mass_function(user_query)
            DS_prediction = ds_calculator.evaluate_with_clarifications(initial_mass)
            results.append({
                "query": user_query,
                "prediction": str(DS_prediction),
                "true_intent": true_intent,
                "interaction": ds_calculator.conversation_history
            })

            print(ds_calculator.conversation_history)

        # Save chunk results
        df = pd.DataFrame(results)
        df.to_csv(f"predictions_test_chunk_{idx}_logistic_B77.csv", index=False)
        print(f"Saved chunk {idx} to predictions_test_chunk_{idx}_logistic_B77.csv")

if __name__ == "__main__":
    main()

[{'text': 'How do I locate my card?', 'label': 11}, {'text': 'I still have not received my new card, I ordered over a week ago.', 'label': 11}, {'text': 'I ordered a card but it has not arrived. Help please!', 'label': 11}, {'text': 'Is there a way to know when my card will arrive?', 'label': 11}, {'text': 'My card has not arrived yet.', 'label': 11}, {'text': 'When will I get my card?', 'label': 11}, {'text': 'Do you know if there is a tracking number for the new card you sent me?', 'label': 11}, {'text': 'i have not received my card', 'label': 11}, {'text': 'still waiting on that card', 'label': 11}, {'text': 'Is it normal to have to wait over a week for my new card?', 'label': 11}, {'text': 'How do I track my card?', 'label': 11}, {'text': 'How long does a card delivery take?', 'label': 11}, {'text': "I still don't have my card after 2 weeks.  What should I do?", 'label': 11}, {'text': 'still waiting on my new card', 'label': 11}, {'text': 'I am still waiting for my card after 1 wee

## Compute accuracy

In [ ]:
import pandas as pd
from sklearn.metrics import f1_score
import re
import numpy as np
import os

def compute_accuracy_from_chunks():
    # Generate file paths for chunks 0 to 9
    base_filename= "predictions_test_chunk_{}_logistic_B77.csv"
    chunk_files = [base_filename.format(i) for i in range(10)]
    dfs = [pd.read_csv(file) for file in chunk_files]
    merged_df = pd.concat(dfs, ignore_index=True)
    merged_df.to_csv("predictions_b77_logistic.csv", index=False)
    # Check if files exist
    missing_files = [f for f in chunk_files if not os.path.exists(f)]
    if missing_files:
        print(f"Error: Missing chunk files: {missing_files}")
    if len(chunk_files) != 10:
        print(f"Warning: Expected 10 chunk files, found {len(chunk_files)}")

    # Parse prediction
    def parse_prediction(pred_str):
        if not isinstance(pred_str, str):
            return "unknown", None
        try:
            # Extract intent and probability from ('intent', np.float64(prob))
            match = re.match(r"\('([^']+)',\s*np\.float64\(([\d.]+)\)\)", pred_str)
            if match:
                intent = match.group(1).lower().strip()
                prob = float(match.group(2))
                return intent, prob
            return "unknown", None
        except Exception as e:
            print(f"Warning: Failed to parse prediction: {pred_str}, error: {e}")
            return "unknown", None

    # Aggregate results
    all_predictions = []
    all_true_labels = []
    all_correct_probs = []
    all_incorrect_probs = []
    total_correct = 0
    total = 0
    per_chunk_metrics = []

    # Process each chunk
    for chunk_file in chunk_files:
        print(f"\nProcessing chunk: {chunk_file}")
        try:
            df = pd.read_csv(chunk_file)
        except FileNotFoundError:
            print(f"Error: {chunk_file} not found")
            continue
        except pd.errors.EmptyDataError:
            print(f"Error: {chunk_file} is empty")
            continue

        # Verify columns
        expected_columns = ["query", "prediction", "true_intent"]
        if not all(col in df.columns for col in expected_columns):
            print(f"Error: {chunk_file} missing required columns. Found: {list(df.columns)}")
            continue

        # Track chunk-specific metrics
        chunk_predictions = []
        chunk_true_labels = []
        chunk_correct_probs = []
        chunk_incorrect_probs = []
        chunk_correct = 0
        chunk_total = 0
        predicted_counts = {}
        true_counts = {}

        # Debug: print first few rows
        print(f"First 5 rows of {chunk_file}:")
        print(df.head().to_string())
        print()

        # Process rows
        for index, row in df.iterrows():
            predicted_intent, prob = parse_prediction(row["prediction"])
            true_intent = str(row["true_intent"]).lower().strip()

            chunk_total += 1
            total += 1
            chunk_predictions.append(predicted_intent)
            chunk_true_labels.append(true_intent)
            all_predictions.append(predicted_intent)
            all_true_labels.append(true_intent)

            # Update counts
            predicted_counts[predicted_intent] = predicted_counts.get(predicted_intent, 0) + 1
            true_counts[true_intent] = true_counts.get(true_intent, 0) + 1

            # Compare
            is_correct = predicted_intent == true_intent
            if is_correct:
                chunk_correct += 1
                total_correct += 1
                if prob is not None:
                    chunk_correct_probs.append(prob)
                    all_correct_probs.append(prob)
            else:
                if prob is not None:
                    chunk_incorrect_probs.append(prob)
                    all_incorrect_probs.append(prob)

            # Debug: print first 10 and matches
            if index < 10 or is_correct:
                print(f"Row {index}: Predicted: {predicted_intent}, True: {true_intent}, Prob: {prob}, Match: {is_correct}")

        # Debug: print counts
        print(f"\nPredicted intent counts in {chunk_file}:")
        for intent, count in sorted(predicted_counts.items()):
            print(f"{intent}: {count}")
        print(f"\nTrue intent counts in {chunk_file}:")
        for intent, count in sorted(true_counts.items()):
            print(f"{intent}: {count}")

        # Compute chunk metrics
        chunk_accuracy = chunk_correct / chunk_total if chunk_total > 0 else 0.0
        chunk_f1 = f1_score(chunk_true_labels, chunk_predictions, average='macro') if chunk_predictions else 0.0
        chunk_avg_correct_prob = np.mean(chunk_correct_probs) if chunk_correct_probs else 0.0
        chunk_avg_incorrect_prob = np.mean(chunk_incorrect_probs) if chunk_incorrect_probs else 0.0

        print(f"\nChunk Metrics for {chunk_file}:")
        print(f"Accuracy: {chunk_accuracy:.4f} ({chunk_correct}/{chunk_total} correct)")
        print(f"Macro-averaged F1 Score: {chunk_f1:.4f}")
        print(f"Average probability for correct predictions: {chunk_avg_correct_prob:.4f} ({len(chunk_correct_probs)} examples)")
        print(f"Average probability for incorrect predictions: {chunk_avg_incorrect_prob:.4f} ({len(chunk_incorrect_probs)} examples)")

        per_chunk_metrics.append({
            'file': chunk_file,
            'accuracy': chunk_accuracy,
            'f1': chunk_f1,
            'correct_prob': chunk_avg_correct_prob,
            'incorrect_prob': chunk_avg_incorrect_prob,
            'correct_count': chunk_correct,
            'total': chunk_total
        })

    # Compute overall metrics
    overall_accuracy = total_correct / total if total > 0 else 0.0
    overall_f1 = f1_score(all_true_labels, all_predictions, average='macro') if all_predictions else 0.0
    overall_avg_correct_prob = np.mean(all_correct_probs) if all_correct_probs else 0.0
    overall_avg_incorrect_prob = np.mean(all_incorrect_probs) if all_incorrect_probs else 0.0

    print("\nOverall Metrics Across All Chunks:")
    print(f"Total utterances processed: {total}")
    print(f"Accuracy: {overall_accuracy:.4f} ({total_correct}/{total} correct)")
    print(f"Macro-averaged F1 Score: {overall_f1:.4f}")
    print(f"Average probability for correct predictions: {overall_avg_correct_prob:.4f} ({len(all_correct_probs)} examples)")
    print(f"Average probability for incorrect predictions: {overall_avg_incorrect_prob:.4f} ({len(all_incorrect_probs)} examples)")

    # Summary of per-chunk metrics
    print("\nPer-Chunk Summary:")
    for metric in per_chunk_metrics:
        print(f"{metric['file']}: Accuracy={metric['accuracy']:.4f}, F1={metric['f1']:.4f}, "
              f"Correct Prob={metric['correct_prob']:.4f}, Incorrect Prob={metric['incorrect_prob']:.4f}")

compute_accuracy_from_chunks()


# Analysis of results

In [ ]:
import pandas as pd

merged_file= "predictions_b77_logistic.csv"
# Read the CSV (only the 'interaction' column)
df = pd.read_csv(merged_file, usecols=['interaction'])

# Count occurrences of "chatbot:" and track rows
total_count = sum(str(interaction).count("Chatbot:") for interaction in df['interaction'] if pd.notna(interaction))
num_rows = sum(1 for interaction in df['interaction'] if pd.notna(interaction))

# Calculate average
average_per_row = total_count / num_rows if num_rows > 0 else 0

# Print results
print(f"Total occurrences of 'chatbot:': {total_count}")
print(f"Rows processed: {num_rows}")
print(f"Average 'chatbot:' per row: {average_per_row:.4f}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from statsmodels.stats.contingency_tables import mcnemar
predictions = clf.predict(test_embeddings)

# Load the dataset from the CSV file (the one you mentioned earlier)
data = pd.read_csv('predictions_b77_logistic.csv')

# Extract true labels and model predictions
true_labels = data['true_intent'].values
print(true_labels[0])
# Manually encode the true labels to numeric values using the label_to_int dictionary

# Extract your model's predictions from the CSV (first element of the tuple in 'prediction' column)
model_predictions = data['prediction'].apply(lambda x: eval(x)[0]).values
print(model_predictions[0])
# Manually encode the model predictions to numeric values using the label_to_int dictionary
# Make predictions using the logistic model
logistic_predictions = clf.predict(test_embeddings)
#logistic_predictions = [index_to_name[idx] for idx in logistic_predictions]
print(logistic_predictions[0])

# Create the contingency table for the McNemar test
# b = Number of instances where SVM is correct and your model is wrong
# c = Number of instances where your model is correct and SVM is wrong
b = np.sum((logistic_predictions == true_labels) & (model_predictions!= true_labels))
c = np.sum((logistic_predictions != true_labels) & (model_predictions == true_labels))

# Construct the contingency table
contingency_table = np.array([[0, b], [c, 0]])

# Perform McNemar test
result = mcnemar(contingency_table, exact=True)

# Print the result
print(f"McNemar test result: statistic={result.statistic}, p-value={result.pvalue}")
